# Data Cleaning and Transformation Notebook

## Purpose

This notebook converts raw CSV data into a structured dataset suitable for analytical processing.

Transformations performed:

- Handling missing values
- Converting date fields
- Converting numeric fields
- Converting boolean fields
- Snake case column conversion
- Preparing the dataset for Delta Lake storage

Output table:

clean_exim_data

In [17]:
from pyspark.sql.functions import col, when

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 36, Finished, Available, Finished, False)

In [18]:
df = spark.read.format("csv") \
    .option("header","true") \
    .load("Files/raw/Exim Financial Dataset.csv")

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 37, Finished, Available, Finished, False)

In [19]:
df = df.replace("N/A", None)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 38, Finished, Available, Finished, False)

In [20]:
df = df.withColumn(
"Fiscal Year",
col("Fiscal Year").cast("int")
)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 39, Finished, Available, Finished, False)

In [21]:
df = df.withColumn(
"Decision Date",
col("Decision Date").cast("date")
).withColumn(
"Effective Date",
col("Effective Date").cast("date")
).withColumn(
"Expiration Date",
col("Expiration Date").cast("date")
)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 40, Finished, Available, Finished, False)

In [23]:
df = df.withColumn(
"Brokered ",
when(col("Brokered ") == "Yes", True).otherwise(False)
)

df = df.withColumn(
"Deal Cancelled",
when(col("Deal Cancelled") == "Yes", True).otherwise(False)
)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 42, Finished, Available, Finished, False)

In [24]:
df = df.withColumn(
"Approved Amount",
col("Approved/Declined Amount").cast("double")
)

df = df.withColumn(
"Disbursed Amount",
col("Disbursed/Shipped Amount").cast("double")
)

df = df.withColumn(
"Outstanding Exposure",
col("Outstanding Exposure Amount").cast("double")
)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 43, Finished, Available, Finished, False)

In [25]:
df = df.withColumn(
"Loan Interest Rate",
col("Loan Interest Rate").cast("double")
)

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 44, Finished, Available, Finished, False)

In [26]:
df.printSchema()

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 45, Finished, Available, Finished, False)

root
 |-- Fiscal Year: integer (nullable = true)
 |-- Unique Identifier: string (nullable = true)
 |-- Deal Number: string (nullable = true)
 |-- Decision: string (nullable = true)
 |-- Decision Date: date (nullable = true)
 |-- Effective Date: date (nullable = true)
 |-- Expiration Date: date (nullable = true)
 |-- Brokered : boolean (nullable = false)
 |-- Deal Cancelled: boolean (nullable = false)
 |-- Country: string (nullable = true)
 |-- Program: string (nullable = true)
 |-- Policy Type: string (nullable = true)
 |-- Decision Authority: string (nullable = true)
 |-- Primary Export Product NAICS/SIC code: string (nullable = true)
 |-- Product Description: string (nullable = true)
 |-- Term: string (nullable = true)
 |-- Primary Applicant: string (nullable = true)
 |-- Primary Lender: string (nullable = true)
 |-- Primary Exporter: string (nullable = true)
 |-- Primary Exporter City: string (nullable = true)
 |-- Primary Exporter State Code: string (nullable = true)
 |-- Primary E

In [27]:
import re

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 46, Finished, Available, Finished, False)

In [28]:
def to_snake_case(column_name):
    
    column_name = column_name.strip()
    
    column_name = column_name.lower()
    
    column_name = re.sub(r"[ /()\-]+", "_", column_name)
    
    column_name = re.sub(r"__+", "_", column_name)
    
    return column_name

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 47, Finished, Available, Finished, False)

In [29]:
df = df.toDF(*[to_snake_case(c) for c in df.columns])

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 48, Finished, Available, Finished, False)

In [30]:
df.printSchema()

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 49, Finished, Available, Finished, False)

root
 |-- fiscal_year: integer (nullable = true)
 |-- unique_identifier: string (nullable = true)
 |-- deal_number: string (nullable = true)
 |-- decision: string (nullable = true)
 |-- decision_date: date (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- expiration_date: date (nullable = true)
 |-- brokered: boolean (nullable = false)
 |-- deal_cancelled: boolean (nullable = false)
 |-- country: string (nullable = true)
 |-- program: string (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- decision_authority: string (nullable = true)
 |-- primary_export_product_naics_sic_code: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- term: string (nullable = true)
 |-- primary_applicant: string (nullable = true)
 |-- primary_lender: string (nullable = true)
 |-- primary_exporter: string (nullable = true)
 |-- primary_exporter_city: string (nullable = true)
 |-- primary_exporter_state_code: string (nullable = true)
 |-- primary_ex

In [31]:
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("clean_exim_data")

StatementMeta(, 299f601e-eb3a-45f5-9553-85a3b275d5df, 50, Finished, Available, Finished, False)